In [1]:
import nest_asyncio
nest_asyncio.apply()

import pyshark
import pandas as pd


In [2]:

cap = pyshark.FileCapture('from_4_to_10_5003_port.pcapng')

data = []
for packet in cap:
    try:
        # Добавляем также payload, если есть
        payload = None
        if hasattr(packet, 'tcp') and hasattr(packet.tcp, 'payload'):
            payload = packet.tcp.payload.replace(':', '')  # убираем двоеточия
        elif hasattr(packet, 'udp') and hasattr(packet.udp, 'payload'):
            payload = packet.udp.payload.replace(':', '')
        data.append({
            'no': packet.number,
            'time': packet.sniff_time,
            'length': int(packet.length),
            'src_ip': packet.ip.src if hasattr(packet, 'ip') else None,
            'dst_ip': packet.ip.dst if hasattr(packet, 'ip') else None,
            'protocol': packet.transport_layer if hasattr(packet, 'transport_layer') else None,
            'info': packet.info if hasattr(packet, 'info') else None,
            'payload': payload,
        })
    except AttributeError:
        continue

df = pd.DataFrame(data)

In [3]:
df

,no,time,length,src_ip,dst_ip,protocol,info,payload
0,1,2026-07-06 18:21:18.580933,58,192.168.1.1,192.168.1.2,TCP,None,5aa58150
1,2,2026-07-06 18:21:18.580998,54,192.168.1.2,192.168.1.1,TCP,None,NaN
2,3,2026-07-06 18:21:18.599721,58,192.168.1.2,192.168.1.1,TCP,None,5aa5a150
3,4,2026-07-06 18:21:18.635870,58,192.168.1.1,192.168.1.2,TCP,None,5aa58150
4,5,2026-07-06 18:21:18.635987,54,192.168.1.2,192.168.1.1,TCP,None,NaN
...,...,...,...,...,...,...,...,...
350,351,2026-07-06 18:21:26.414589,54,192.168.1.1,192.168.1.2,TCP,None,NaN
351,352,2026-07-06 18:21:26.461117,74,192.168.1.1,192.168.1.2,TCP,None,5bb5000000000000000000000000000000000000
352,353,2026-07-06 18:21:26.461181,54,192.168.1.2,192.168.1.1,TCP,None,NaN
353,354,2026-07-06 18:21:26.462469,58,192.168.1.1,192.168.1.2,TCP,None,5aa58150


In [4]:
(df['payload'].unique())

<StringArray>
[                                                                '5aa58150',
                                                                        nan,
                                                                 '5aa5a150',
                                 '5bb5000000000000000000000000000000000000',
                                                                 '5aa5ad50',
 '5aa5ad505ff5000000005000000000030000005959950000400057ff2801500000000003']
Length: 6, dtype: str

In [5]:
df['payload'].value_counts()

payload
5aa58150                                                                    135
5aa5a150                                                                     56
5aa5ad50                                                                      5
5bb5000000000000000000000000000000000000                                      4
5aa5ad505ff5000000005000000000030000005959950000400057ff2801500000000003      1
Name: count, dtype: int64

5aa5ad505ff5000000005000000000030000005959950000400057ff2801500000000003